In [2]:
# ===== PHẦN 1: IMPORT LIBRARY + LOAD DATA THÔ (tên file thật) =====

import pandas as pd
import numpy as np
import os

# Đường dẫn gốc chứa data thô — XÁC NHẬN LẠI cho đúng
RAW_DATA_PATH = r"D:\KLTN\Data Dictionary\Modeling\Data"

# Đường dẫn folder xuất file đã clean
OUTPUT_PATH = r"D:\KLTN\Data Dictionary\Model - Data used"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Load các file theo đúng tên thật trong folder
customer_master = pd.read_parquet(os.path.join(RAW_DATA_PATH, "CustomerMaster.parquet"))
b2b_so = pd.read_parquet(os.path.join(RAW_DATA_PATH, "B2B_SO_2025_JantoNov.parquet"))
demand_alloc_b2b = pd.read_parquet(os.path.join(RAW_DATA_PATH, "ENO_DemandAllocationB2B.parquet"))
product_master = pd.read_parquet(os.path.join(RAW_DATA_PATH, "ProductMaster.parquet"))

# TODO: facility_master — chưa rõ tên file, có thể nằm trong folder "0,1_Baseline2025"
# facility_master = pd.read_parquet(os.path.join(RAW_DATA_PATH, "0,1_Baseline2025", "???.parquet"))

# Kiểm tra nhanh
print("customer_master:", customer_master.shape)
print("b2b_so:", b2b_so.shape)
print("demand_alloc_b2b:", demand_alloc_b2b.shape)
print("product_master:", product_master.shape)

print("\n--- customer_master ---")
print(customer_master.head(10))
print("\n--- b2b_so ---")
print(b2b_so.head(10))
print("\n--- demand_alloc_b2b ---")
print(demand_alloc_b2b.head(10))
print("\n--- product_master ---")
print(product_master.head(10))

customer_master: (48001, 16)
b2b_so: (25260049, 27)
demand_alloc_b2b: (5835102, 6)
product_master: (5174, 31)

--- customer_master ---
   LocationID Type LocationType  CentroidLong  CentroidLat          Channel  \
0  120173-002  B2B     ShipToID    115.214557    -8.665922     Modern Trade   
1  162179-000  B2B     ShipToID    115.338667    -8.548140  Health & Beauty   
2  134882-000  B2B     ShipToID    115.013533    -8.216141    General Trade   
3  154164-000  B2B     ShipToID    115.216954    -8.673516           Others   
4  138046-000  B2B     ShipToID    115.510991    -8.501664    General Trade   
5  157128-000  B2B     ShipToID    114.834975    -8.424671    General Trade   
6  102328-000  B2B     ShipToID    115.215858    -8.621308    General Trade   
7  167727-000  B2B     ShipToID    115.214592    -8.542390    General Trade   
8  167609-000  B2B     ShipToID    115.614362    -8.450825    General Trade   
9  140103-000  B2B     ShipToID    115.176602    -8.724630     Modern Trade

# 1. Customer Master

In [3]:
# ===== PHẦN 2: CLEAN customer_master =====

# Kiểm tra lại vấn đề: LocationID "012019121151" (Legok, Tangerang, Banten)
# bị geocode sai sang tọa độ ngoài Java (Sulawesi)
check_before = customer_master[customer_master["LocationID"] == "012019121151"]
print("Trước khi fix:")
print(check_before[["LocationID", "District", "City", "CentroidLat", "CentroidLong"]])

# Fix: gán lại tọa độ đúng, lấy từ khách hàng khác cùng District (Legok)
# đã xác nhận tọa độ -6.30 / 106.57 là hợp lý cho khu vực Tangerang
customer_master.loc[customer_master["LocationID"] == "012019121151", "CentroidLat"] = -6.30
customer_master.loc[customer_master["LocationID"] == "012019121151", "CentroidLong"] = 106.57

# Kiểm tra lại sau khi fix
check_after = customer_master[customer_master["LocationID"] == "012019121151"]
print("\nSau khi fix:")
print(check_after[["LocationID", "District", "City", "CentroidLat", "CentroidLong"]])

# head(10) tổng thể để soát lại toàn bảng
print("\nHead(10) customer_master sau clean:")
print(customer_master.head(10))
print(f"\nShape: {customer_master.shape}")

Trước khi fix:
         LocationID District       City  CentroidLat  CentroidLong
46146  012019121151    Legok  Tangerang     1.460532    124.930054

Sau khi fix:
         LocationID District       City  CentroidLat  CentroidLong
46146  012019121151    Legok  Tangerang         -6.3        106.57

Head(10) customer_master sau clean:
   LocationID Type LocationType  CentroidLong  CentroidLat          Channel  \
0  120173-002  B2B     ShipToID    115.214557    -8.665922     Modern Trade   
1  162179-000  B2B     ShipToID    115.338667    -8.548140  Health & Beauty   
2  134882-000  B2B     ShipToID    115.013533    -8.216141    General Trade   
3  154164-000  B2B     ShipToID    115.216954    -8.673516           Others   
4  138046-000  B2B     ShipToID    115.510991    -8.501664    General Trade   
5  157128-000  B2B     ShipToID    114.834975    -8.424671    General Trade   
6  102328-000  B2B     ShipToID    115.215858    -8.621308    General Trade   
7  167727-000  B2B     ShipToID   

In [4]:
# ===== XUẤT customer_master ĐÃ CLEAN RA PARQUET =====

output_file = os.path.join(OUTPUT_PATH, "customer_master_clean.parquet")
customer_master.to_parquet(output_file, index=False)

print(f"Đã lưu: {output_file}")
print(f"Shape: {customer_master.shape}")

# Đọc lại kiểm tra file lưu đúng
check = pd.read_parquet(output_file)
print("\nHead(10) sau khi đọc lại từ file:")
print(check.head(10))

Đã lưu: D:\KLTN\Data Dictionary\Model - Data used\customer_master_clean.parquet
Shape: (48001, 16)

Head(10) sau khi đọc lại từ file:
   LocationID Type LocationType  CentroidLong  CentroidLat          Channel  \
0  120173-002  B2B     ShipToID    115.214557    -8.665922     Modern Trade   
1  162179-000  B2B     ShipToID    115.338667    -8.548140  Health & Beauty   
2  134882-000  B2B     ShipToID    115.013533    -8.216141    General Trade   
3  154164-000  B2B     ShipToID    115.216954    -8.673516           Others   
4  138046-000  B2B     ShipToID    115.510991    -8.501664    General Trade   
5  157128-000  B2B     ShipToID    114.834975    -8.424671    General Trade   
6  102328-000  B2B     ShipToID    115.215858    -8.621308    General Trade   
7  167727-000  B2B     ShipToID    115.214592    -8.542390    General Trade   
8  167609-000  B2B     ShipToID    115.614362    -8.450825    General Trade   
9  140103-000  B2B     ShipToID    115.176602    -8.724630     Modern Trade 

# 2. Demand Allocation

In [5]:
# ===== PHẦN 4: CLEAN b2b_so =====

# Lưu số liệu trước khi clean để so sánh
before_shape = b2b_so.shape

# Kiểm tra lại các dòng rác: QtyOrderedInKG <= 0 (đơn hàng hủy/placeholder)
junk_rows = b2b_so[b2b_so["QtyOrderedInKG"] <= 0]
print(f"Số dòng QtyOrderedInKG <= 0: {junk_rows.shape[0]}")
print("\nHead(10) dòng rác trước khi bỏ:")
print(junk_rows.head(10))

# Bỏ dòng rác (QtyOrderedInKG <= 0) và dòng duplicate hoàn toàn
b2b_so = b2b_so[b2b_so["QtyOrderedInKG"] > 0].drop_duplicates()

# So sánh trước/sau
print(f"\nShape trước: {before_shape}")
print(f"Shape sau: {b2b_so.shape}")
print(f"Đã loại: {before_shape[0] - b2b_so.shape[0]:,} dòng")

# head(10) tổng thể để soát lại toàn bảng
print("\nHead(10) b2b_so sau clean:")
print(b2b_so.head(10))

Số dòng QtyOrderedInKG <= 0: 1029

Head(10) dòng rác trước khi bỏ:
           SOID   SOLineID PickingID   PickingDocumentName          SONumber  \
48177   8105064  143035124  19702798    PRM/32543600048452  PRM/#SO258103695   
50874      NULL       NULL  19572183    PRM/32541100000829              NULL   
54911      NULL       NULL  20478006    PRM/32532000027329              NULL   
60112      NULL       NULL  18607512    PRM/32543300002297              NULL   
70881   8346613  147734660  20491210   PRM/C32540500000049  PRM/#SO258345245   
101210  8105064  143035148  19702798    PRM/32543600048452  PRM/#SO258103695   
113705     NULL       NULL  19572183    PRM/32541100000829              NULL   
130930     NULL       NULL  19572183    PRM/32541100000829              NULL   
190698     NULL       NULL  20468496    PRM/32542600009408              NULL   
246501     NULL       NULL  18832166  PRM/MW32520900000028              NULL   

       OrderStatus          OrderedTimestamp     Con

In [6]:
output_file = os.path.join(OUTPUT_PATH, "b2b_so_clean.parquet")
b2b_so.to_parquet(output_file, index=False)

# 3. Product 

In [8]:
# Check Case A/B cho nhóm KGPerPallet>2000
case_a_high = high_group[high_group["QuantityPiecesPerPallet"].fillna(1) <= 1]
case_b_high = high_group[high_group["QuantityPiecesPerPallet"].fillna(1) > 1]
print(f"KGPerPallet>2000: Case A: {case_a_high.shape[0]}, Case B: {case_b_high.shape[0]}")

# Đo mức ảnh hưởng thật (kg) của toàn bộ 2 nhóm lỗi còn lại (132 + 208 = 340 sản phẩm)
problem_ids = set(low_group["ProductID"]) | set(high_group["ProductID"])
problem_orders = b2b_so[b2b_so["ProductID"].isin(problem_ids)]
print(f"\nTổng {len(problem_ids)} sản phẩm lỗi còn lại")
print(f"  → {problem_orders.shape[0]:,} dòng đơn hàng thật")
print(f"  → {problem_orders['QtyOrderedInKG'].sum():,.0f} kg ({problem_orders['QtyOrderedInKG'].sum()/total_kg_all*100:.3f}% tổng network)")

KGPerPallet>2000: Case A: 0, Case B: 208

Tổng 340 sản phẩm lỗi còn lại
  → 88,497 dòng đơn hàng thật
  → 502,628 kg (1.703% tổng network)


In [10]:
# ===== LOẠI BỎ 340 SẢN PHẨM CÒN LỖI KGPerPallet =====
# Quyết định: bỏ hẳn thay vì fix, vì chỉ chiếm 1.7% kg toàn network — chấp nhận được

LOW_THRESHOLD = 10
HIGH_THRESHOLD = 2000

# Xác định danh sách sản phẩm lỗi cần loại (KGPerPallet < 10 hoặc > 2000)
problem_ids = set(product_master_scoped[
    (product_master_scoped["KGPerPallet"] < LOW_THRESHOLD) | (product_master_scoped["KGPerPallet"] > HIGH_THRESHOLD)
]["ProductID"])

# Lưu số liệu trước khi loại để so sánh
before_pm = product_master_scoped.shape[0]
before_so = b2b_so.shape[0]
before_kg = b2b_so["QtyOrderedInKG"].sum()

# Loại khỏi product_master
product_master = product_master_scoped[~product_master_scoped["ProductID"].isin(problem_ids)].copy()

# Loại đơn hàng tương ứng khỏi b2b_so để đồng bộ
b2b_so = b2b_so[~b2b_so["ProductID"].isin(problem_ids)].copy()

# So sánh trước/sau
print(f"product_master: {before_pm} → {product_master.shape[0]} (loại {len(problem_ids)} sản phẩm)")
print(f"b2b_so: {before_so:,} → {b2b_so.shape[0]:,} dòng")
print(f"Tổng kg: {before_kg:,.0f} → {b2b_so['QtyOrderedInKG'].sum():,.0f} kg "
      f"(loại {(before_kg - b2b_so['QtyOrderedInKG'].sum())/before_kg*100:.2f}%)")

# Xác nhận không còn sản phẩm lỗi nào
remaining_bad = product_master[(product_master["KGPerPallet"] < LOW_THRESHOLD) | (product_master["KGPerPallet"] > HIGH_THRESHOLD)]
print(f"\nSố sản phẩm còn lỗi KGPerPallet: {remaining_bad.shape[0]}")

print("\nHead(10) product_master sau khi loại:")
print(product_master.head(10))

product_master: 4957 → 4617 (loại 340 sản phẩm)
b2b_so: 25,258,999 → 25,170,502 dòng
Tổng kg: 29,508,705 → 29,006,077 kg (loại 1.70%)

Số sản phẩm còn lỗi KGPerPallet: 0

Head(10) product_master sau khi loại:
  ProductID ProductBarcode     CorporateCode  GrossWeightInGram  \
0     02638  8993137690829     MY-RBBCLLN1-W              43.50   
1     06384  8993137685610      TC-R5LLF03-W              62.01   
2     00127  8993137690881      BBC-EL-W30ML              45.60   
3     08406  8993137007634       L-R1ELV10-W              35.35   
4     05760  8993137740258       MY-LCLADF-W              29.48   
5     08399  8993137000680       L-R1ELV03-W              35.35   
6     00184  8993137679800  L-LLW0C10-W2,5GR              15.50   
7     03372  8993137714174        L-CLCLS1-W              26.63   
8     00222  8993137693578       PUFF-W0N-WP               1.00   
9     04779  8993137679053       TC-R2RE01-W              31.50   

   NetWeightInGram         CBM  UnitBoxLengthMm  Unit

In [11]:
# ===== XUẤT product_master VÀ b2b_so (BẢN CUỐI) RA PARQUET =====

pm_output = os.path.join(OUTPUT_PATH, "product_master_clean.parquet")
so_output = os.path.join(OUTPUT_PATH, "b2b_so_final.parquet")

product_master.to_parquet(pm_output, index=False)
b2b_so.to_parquet(so_output, index=False)

In [12]:
# Dùng thẳng file đã có sẵn (không cần clean lại vì đã sạch 100% từ trước)
facility_master = pd.read_parquet(r"D:\KLTN\Data Dictionary\Model - Data used\FacilityMaster.parquet")

# Kiểm tra cơ bản để xác nhận vẫn đúng như trước
print("Shape:", facility_master.shape)
print("\nNulls:")
print(facility_master.isnull().sum()[facility_master.isnull().sum() > 0])

print("\nCác cột:", facility_master.columns.tolist())

print("\nHead(10):")
print(facility_master.head(10))

if "FacilityType" in facility_master.columns:
    print("\nFacilityType breakdown:")
    print(facility_master["FacilityType"].value_counts())

Shape: (67, 20)

Nulls:
Series([], dtype: int64)

Các cột: ['LocationID', 'FacilityName', 'FacilityType', 'MainIsland', 'CapacityInPallet', 'AreaSQM', 'Status', 'Latitude', 'Longitude', 'Country', 'FixedStorageCost', 'B2BPackagingCostPerPallet', 'B2BManpowerCostPerPallet', 'B2CPackagingCostPerPCS', 'B2CManpowerCostPerPCS', 'MonthsPresentIn2025', 'FixedStorageCost_perMonth', 'B2BHandlingCostPerPallet', 'B2CHandlingCostPerPCS', 'StoragePenaltyPerPallet']

Head(10):
  LocationID         FacilityName  FacilityType    MainIsland  \
0        D03    DC Satellite Bali  DC Satellite  Bali - Nusra   
1        D06      RDC Banjarmasin           RDC    Kalimantan   
2        D09      DC Direct Bogor     DC Direct          Java   
3        D10          RDC Cirebon           RDC          Java   
4        D16            RDC Medan           RDC       Sumatra   
5        D18        RDC Palembang           RDC       Sumatra   
6        D20  DC Direct Pontianak     DC Direct    Kalimantan   
7        D25

In [14]:
# ===== LỌC facility_master THEO SCOPE JAVA (khớp với phạm vi khách hàng) =====

facility_master = facility_master[facility_master["MainIsland"] == "Java"].copy()

print("Shape sau khi lọc Java:", facility_master.shape)
print(facility_master["FacilityType"].value_counts())

# ===== XUẤT facility_master (SCOPE JAVA) RA PARQUET =====

fm_output = os.path.join(OUTPUT_PATH, "FacilityMaster.parquet")
facility_master.to_parquet(fm_output, index=False)

Shape sau khi lọc Java: (27, 20)
FacilityType
DEPO            9
DC Direct       5
RDC             4
FC              3
DC Satellite    3
Instant Hub     2
NDC             1
Name: count, dtype: int64


In [19]:
# ===== TẠO shipto_crosswalk (nếu chưa có) TỪ demand_alloc_b2b =====

shipto_crosswalk = demand_alloc_b2b[["ShipToID", "ShipToGroupID"]].drop_duplicates()

# ===== LỌC customer_master + b2b_so + shipto_crosswalk THEO SCOPE JAVA =====

customer_master = customer_master[customer_master["MainIsland"] == "Java"].copy()

java_shipto_ids = set(customer_master["LocationID"])
b2b_so = b2b_so[b2b_so["ShipToID"].isin(java_shipto_ids)].copy()
shipto_crosswalk = shipto_crosswalk[shipto_crosswalk["ShipToID"].isin(java_shipto_ids)].copy()

# ===== XUẤT LẠI customer_master, b2b_so, shipto_crosswalk (BẢN JAVA) =====

customer_master.to_parquet(os.path.join(OUTPUT_PATH, "customer_master_clean.parquet"), index=False)
b2b_so.to_parquet(os.path.join(OUTPUT_PATH, "b2b_so_final.parquet"), index=False)
shipto_crosswalk.to_parquet(os.path.join(OUTPUT_PATH, "shipto_crosswalk.parquet"), index=False)

In [ ]:
# ===== FIX: loại 148 ProductID ngoài scope còn sót lại trong b2b_so =====
# Nguyên nhân: bước filter Segment ở Phần 5a chưa áp đúng cho b2b_so,
# nên vẫn còn sót order của product Lifestyle/Health & Wellness/Segment=NaN

# Lấy tập ProductID cuối cùng từ product_master (đã scope đầy đủ)
final_product_ids = set(product_master["ProductID"])

# Lọc lại b2b_so, chỉ giữ order thuộc đúng scope
b2b_so = b2b_so[b2b_so["ProductID"].isin(final_product_ids)].copy()

# Xuất lại b2b_so_final.parquet (bản đã fix, không còn ProductID ngoài scope)
b2b_so.to_parquet(os.path.join(OUTPUT_PATH, "b2b_so_final.parquet"), index=False)

In [ ]:
# ===== RE-CHECK: xác nhận đã hết orphan ProductID =====
product_in_b2b_so = set(b2b_so["ProductID"])
orphan_product_after_fix = product_in_b2b_so - final_product_ids
print("Số Prod  uctID orphan còn lại sau fix:", len(orphan_product_after_fix))

Số ProductID orphan còn lại sau fix: 0


In [2]:
import pandas as pd
import os

RAW_DATA_PATH = r"D:\KLTN\Data Dictionary\Modeling\Data"
OUTPUT_PATH = r"D:\KLTN\Data Dictionary\Model - Data used"

# Load lại demand_alloc_b2b (raw, chưa lọc gì - để lấy số cluster TRƯỚC Java)
demand_alloc_b2b = pd.read_parquet(os.path.join(RAW_DATA_PATH, "ENO_DemandAllocationB2B.parquet"))

# Load lại shipto_crosswalk bản đã clean (đã lọc Java, lấy từ file đã export)
shipto_crosswalk = pd.read_parquet(os.path.join(OUTPUT_PATH, "shipto_crosswalk.parquet"))

# ===== VERIFY: 1,763 có đúng là số cluster sau khi lọc Java hay ko =====
n_clusters_before_java = demand_alloc_b2b["ShipToGroupID"].nunique()
n_clusters_after_java = shipto_crosswalk["ShipToGroupID"].nunique()

print("Số cluster TRƯỚC khi lọc Java (toàn Indonesia + Malaysia):", n_clusters_before_java)
print("Số cluster SAU khi lọc Java (đang dùng cho demand_agg):", n_clusters_after_java)

Số cluster TRƯỚC khi lọc Java (toàn Indonesia + Malaysia): 3433
Số cluster SAU khi lọc Java (đang dùng cho demand_agg): 1763


In [3]:
import pandas as pd
import os

RAW_DATA_PATH = r"D:\KLTN\Data Dictionary\Modeling\Data"

# ===== Load file Unconstrained mới tìm được =====
cost_unconstrained = pd.read_parquet(os.path.join(RAW_DATA_PATH, "ENO_CostLastMileB2B_Unconstaint.parquet"))

# ===== Load lại file cũ (Constrained/Baseline) để so sánh =====
cost_constrained = pd.read_parquet(os.path.join(RAW_DATA_PATH, "0,1_Baseline2025", "ENO_CostLastMileB2B.parquet"))

# ===== So sánh cấu trúc cột giữa 2 file =====
print("===== File CŨ (Constrained/Baseline) =====")
print("Shape:", cost_constrained.shape)
print("Columns:", cost_constrained.columns.tolist())
print(cost_constrained.head(10))

print("\n===== File MỚI (Unconstrained) =====")
print("Shape:", cost_unconstrained.shape)
print("Columns:", cost_unconstrained.columns.tolist())
print(cost_unconstrained.head(10))

# ===== So sánh số lượng Origin/Destination unique giữa 2 file =====
print("\nSố OriginID unique - CŨ:", cost_constrained["OriginID"].nunique(), "| MỚI:", cost_unconstrained["OriginID"].nunique())
print("Số DestinationID unique - CŨ:", cost_constrained["DestinationID"].nunique(), "| MỚI:", cost_unconstrained["DestinationID"].nunique())

# ===== Check coverage: trung bình mỗi Destination có bao nhiêu Origin phục vụ =====
coverage_old = cost_constrained.groupby("DestinationID")["OriginID"].nunique()
coverage_new = cost_unconstrained.groupby("DestinationID")["OriginID"].nunique()

print("\nPhân bố n_facilities_covering - CŨ (Constrained):")
print(coverage_old.describe())

print("\nPhân bố n_facilities_covering - MỚI (Unconstrained):")
print(coverage_new.describe())

===== File CŨ (Constrained/Baseline) =====
Shape: (3635, 3)
Columns: ['OriginID', 'DestinationID', 'CostPerPallet']
  OriginID                  DestinationID  CostPerPallet
0      D03  ID_BLN_Bali_Badung_Abiansemal   2.366712e+05
1      D03        ID_BLN_Bali_Badung_Kuta   2.059546e+05
2      D03      ID_BLN_Bali_Badung_Mengwi   1.916714e+05
3      D03      ID_BLN_Bali_Badung_Petang   6.455088e+05
4      D03      ID_BLN_Bali_Bangli_Bangli   6.418228e+05
5      D03   ID_BLN_Bali_Bangli_Kintamani   9.783228e+05
6      D03       ID_BLN_Bali_Bangli_Susut   7.350476e+05
7      D03     ID_BLN_Bali_Bangli_Tembuku   8.069244e+05
8      D03    ID_BLN_Bali_Buleleng_Banjar   1.123151e+06
9      D03  ID_BLN_Bali_Buleleng_Buleleng   1.187656e+06

===== File MỚI (Unconstrained) =====
Shape: (22852, 3)
Columns: ['OriginID', 'DestinationID', 'CostPerPallet']
  OriginID                  DestinationID  CostPerPallet
0      D03  ID_BLN_Bali_Badung_Abiansemal   3.142993e+05
1      D03        ID_BLN_Bali_B

In [7]:
import pandas as pd
import numpy as np
import os

# ===== PATHS =====
DATA_PATH = r"D:\KLTN\Data Dictionary\Model - Data used"
RAW_DATA_PATH = r"D:\KLTN\Data Dictionary\Modeling\Data"

# ===== Load 5 bảng đã clean (Section 4.1) =====
customer_master = pd.read_parquet(os.path.join(DATA_PATH, "customer_master_clean.parquet"))
b2b_so = pd.read_parquet(os.path.join(DATA_PATH, "b2b_so_final.parquet"))
shipto_crosswalk = pd.read_parquet(os.path.join(DATA_PATH, "shipto_crosswalk.parquet"))
product_master = pd.read_parquet(os.path.join(DATA_PATH, "product_master_clean.parquet"))
facility_master = pd.read_parquet(os.path.join(DATA_PATH, "FacilityMaster.parquet"))

# ===== Rebuild facility_model (Section 4.4) =====
FIXED_TYPES = ["NDC"]
DECISION_TYPES = ["DEPO", "RDC", "DC Direct", "DC Satellite", "Instant Hub"]
EXCLUDED_TYPES = ["FC"]

def classify_facility(facility_type):
    if facility_type in FIXED_TYPES:
        return "FIXED"
    elif facility_type in DECISION_TYPES:
        return "DECISION"
    elif facility_type in EXCLUDED_TYPES:
        return "EXCLUDED"
    else:
        return "UNKNOWN"

facility_master["FacilityRole"] = facility_master["FacilityType"].apply(classify_facility)
facility_model = facility_master[facility_master["FacilityRole"] != "EXCLUDED"].copy()

# ===== Rebuild demand_agg (Section 4.2) =====
b2b_agg = b2b_so.merge(shipto_crosswalk, on="ShipToID", how="left")
b2b_agg = b2b_agg.merge(product_master[["ProductID", "KGPerPallet"]], on="ProductID", how="left")
b2b_agg["QtyInPallet"] = b2b_agg["QtyOrderedInKG"] / b2b_agg["KGPerPallet"]
b2b_agg["OrderMonth"] = b2b_agg["OrderedTimestamp"].dt.to_period("M")

monthly_demand = b2b_agg.groupby(["ShipToGroupID", "OrderMonth"])["QtyInPallet"].sum().reset_index()

all_nodes = shipto_crosswalk["ShipToGroupID"].unique()
all_months = monthly_demand["OrderMonth"].unique()
full_index = pd.MultiIndex.from_product([all_nodes, all_months], names=["ShipToGroupID", "OrderMonth"])
monthly_demand_full = (
    monthly_demand.set_index(["ShipToGroupID", "OrderMonth"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

demand_agg = monthly_demand_full.groupby("ShipToGroupID")["QtyInPallet"].agg(
    d_bar="mean", d_max="max"
).reset_index()
demand_agg["d_hat"] = (demand_agg["d_max"] - demand_agg["d_bar"]).clip(lower=0)

# ===== Load file Unconstrained =====
cost_unconstrained = pd.read_parquet(os.path.join(RAW_DATA_PATH, "ENO_CostLastMileB2B_Unconstaint.parquet"))

# ===== BƯỚC 1: Check facility trong scope có xuất hiện trong file Unconstrained ko =====
facility_ids_in_scope = set(facility_model["LocationID"])
origin_ids_in_unconstrained = set(cost_unconstrained["OriginID"])

print("Số facility trong scope (24) CÓ xuất hiện trong file Unconstrained:",
      len(facility_ids_in_scope & origin_ids_in_unconstrained))
print("Facility trong scope KHÔNG xuất hiện (nếu có):")
print(facility_ids_in_scope - origin_ids_in_unconstrained)

# ===== BƯỚC 2: Lọc đúng scope (facility_model x cluster Java) =====
java_clusters = set(demand_agg["ShipToGroupID"])
cost_unconstrained_scoped = cost_unconstrained[
    cost_unconstrained["OriginID"].isin(facility_ids_in_scope) &
    cost_unconstrained["DestinationID"].isin(java_clusters)
].copy()

print("\nShape sau khi lọc đúng scope:", cost_unconstrained_scoped.shape)

# ===== BƯỚC 3: Re-check coverage trong đúng scope =====
coverage_scoped = cost_unconstrained_scoped.groupby("DestinationID")["OriginID"].nunique()
print("\nPhân bố n_facilities_covering (ĐÚNG SCOPE):")
print(coverage_scoped.describe())

missing_clusters = java_clusters - set(coverage_scoped.index)
print(f"\nSố cluster Java KHÔNG có facility nào cover: {len(missing_clusters)}")

C:\Users\USER\AppData\Local\Temp\ipykernel_9920\1283734190.py:38: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  b2b_agg["OrderMonth"] = b2b_agg["OrderedTimestamp"].dt.to_period("M")


Số facility trong scope (24) CÓ xuất hiện trong file Unconstrained: 22
Facility trong scope KHÔNG xuất hiện (nếu có):
{'D54', 'D49'}

Shape sau khi lọc đúng scope: (10080, 3)

Phân bố n_facilities_covering (ĐÚNG SCOPE):
count    1763.000000
mean        5.717527
std         1.534567
min         1.000000
25%         5.000000
50%         6.000000
75%         7.000000
max         9.000000
Name: OriginID, dtype: float64

Số cluster Java KHÔNG có facility nào cover: 0


In [8]:
# ===== CHECK: CostTransfer và CostSupply - Baseline vs Greenfield (GF_Unconstrained) =====

GF_PATH = os.path.join(RAW_DATA_PATH, "3_GF_Unconstrained")
BL_PATH = os.path.join(RAW_DATA_PATH, "0,1_Baseline2025")

# ===== CostTransfer =====
transfer_baseline = pd.read_parquet(os.path.join(BL_PATH, "ENO_CostTransfer.parquet"))
transfer_gf = pd.read_parquet(os.path.join(GF_PATH, "ENO_CostTransfer.parquet"))

print("===== CostTransfer =====")
print("Shape Baseline:", transfer_baseline.shape, "| Shape Greenfield:", transfer_gf.shape)
print("\nColumns Baseline:", transfer_baseline.columns.tolist())
print(transfer_baseline.head(5))
print("\nColumns Greenfield:", transfer_gf.columns.tolist())
print(transfer_gf.head(5))

# Coverage: mỗi Destination có bao nhiêu Origin khả dĩ (dùng đúng tên cột thật, ko đoán vị trí)
cov_transfer_baseline = transfer_baseline.groupby("DestinationID")["OriginID"].nunique()
cov_transfer_gf = transfer_gf.groupby("DestinationID")["OriginID"].nunique()
print("\nCoverage Baseline:")
print(cov_transfer_baseline.describe())
print("\nCoverage Greenfield:")
print(cov_transfer_gf.describe())

# ===== CostSupply =====
supply_baseline = pd.read_parquet(os.path.join(BL_PATH, "ENO_CostSupply.parquet"))
supply_gf = pd.read_parquet(os.path.join(GF_PATH, "ENO_CostSupply.parquet"))

print("\n===== CostSupply =====")
print("Shape Baseline:", supply_baseline.shape, "| Shape Greenfield:", supply_gf.shape)
print(supply_baseline.head(10))
print(supply_gf.head(10))

===== CostTransfer =====
Shape Baseline: (95, 3) | Shape Greenfield: (3000, 5)

Columns Baseline: ['OriginID', 'DestinationID', 'CostPerPallet']
  OriginID DestinationID  CostPerPallet
0      NDC           D03   8.794118e+05
1      NDC           D06   1.417281e+06
2      NDC           D09   1.737804e+05
3      NDC           D10   4.243882e+05
4      NDC           D16   1.569068e+06

Columns Greenfield: ['OriginID', 'DestinationID', 'DistanceInKM', 'CostPerPallet', 'IsExistingFlow']
  OriginID DestinationID  DistanceInKM  CostPerPallet  IsExistingFlow
0      D03           D79          3.93   1.609915e+04            True
1      D03       MWH_D32        128.44   5.261512e+05            True
2      D06           D81          3.42   1.383439e+04            True
3      D06       MWH_D22        633.59   2.534459e+06            True
4      D06       MWH_D38        200.99   8.039913e+05            True

Coverage Baseline:
count    66.000000
mean      1.439394
std       0.558261
min       1.0000

In [9]:
# ===== Lọc CostTransfer (Greenfield) về đúng scope 22 facility trong facility_model =====
facility_ids_in_scope = set(facility_model["LocationID"])

transfer_final = transfer_gf[
    transfer_gf["OriginID"].isin(facility_ids_in_scope) &
    transfer_gf["DestinationID"].isin(facility_ids_in_scope)
].copy()

print("Shape transfer_final (đã lọc scope):", transfer_final.shape)
print(transfer_final.head(10))

# Coverage sau khi lọc: mỗi facility đích có bao nhiêu facility nguồn khả dĩ chuyển hàng tới
coverage_transfer_final = transfer_final.groupby("DestinationID")["OriginID"].nunique()
print("\nPhân bố coverage (đã lọc scope):")
print(coverage_transfer_final.describe())

# Check facility nào trong scope hoàn toàn ko xuất hiện (cả Origin lẫn Destination) trong bảng Transfer
involved_facilities = set(transfer_final["OriginID"]) | set(transfer_final["DestinationID"])
missing_facilities = facility_ids_in_scope - involved_facilities
print(f"\nFacility trong scope KHÔNG xuất hiện trong CostTransfer: {missing_facilities}")

Shape transfer_final (đã lọc scope): (195, 5)
   OriginID DestinationID  DistanceInKM  CostPerPallet  IsExistingFlow
5       D09        DEPO03         36.82   70694.174250            True
6       D09        DEPO04         32.43   62265.401166            True
7       D10       MWH_D21        150.95  687546.427552            True
19      D25        DEPO18         59.55  114335.634888            True
20      D26        DEPO13        130.50  250559.199880            True
21      D26        DEPO17         79.95  153503.509811            True
22      D30           D26        113.37  516377.201004            True
24      D30        DEPO07         82.88  159129.091847            True
25      D30        DEPO08        100.79  193516.182037            True
33      D49           D54         23.54  551042.117619            True

Phân bố coverage (đã lọc scope):
count    23.000000
mean      8.478261
std       4.336626
min       1.000000
25%       3.500000
50%      12.000000
75%      12.000000
max   

In [10]:
# ===== BƯỚC 1: Cập nhật facility_model - loại Instant Hub =====
FIXED_TYPES = ["NDC"]
DECISION_TYPES = ["DEPO", "RDC", "DC Direct", "DC Satellite"]  # bỏ Instant Hub
EXCLUDED_TYPES = ["FC", "Instant Hub"]

def classify_facility(facility_type):
    if facility_type in FIXED_TYPES:
        return "FIXED"
    elif facility_type in DECISION_TYPES:
        return "DECISION"
    elif facility_type in EXCLUDED_TYPES:
        return "EXCLUDED"
    else:
        return "UNKNOWN"

facility_master["FacilityRole"] = facility_master["FacilityType"].apply(classify_facility)
facility_model = facility_master[facility_master["FacilityRole"] != "EXCLUDED"].copy()
facility_ids_in_scope = set(facility_model["LocationID"])

print("Tổng facility_model (đã loại Instant Hub):", facility_model.shape[0])
print("D49/D54 còn trong scope ko:", "D49" in facility_ids_in_scope, "|", "D54" in facility_ids_in_scope)

# ===== BƯỚC 2: Lọc lại CostTransfer theo scope ĐÃ CẬP NHẬT =====
transfer_final = transfer_gf[
    transfer_gf["OriginID"].isin(facility_ids_in_scope) &
    transfer_gf["DestinationID"].isin(facility_ids_in_scope)
].copy()

print("\nShape transfer_final (scope đã đúng, 22 facility):", transfer_final.shape)
coverage_transfer_final = transfer_final.groupby("DestinationID")["OriginID"].nunique()
print(coverage_transfer_final.describe())

# ===== BƯỚC 3: Lọc lại CostLastMileB2B theo scope ĐÃ CẬP NHẬT (đề phòng cũng bị lẫn D49/D54) =====
java_clusters = set(demand_agg["ShipToGroupID"])
cost_lastmile_final = cost_unconstrained[
    cost_unconstrained["OriginID"].isin(facility_ids_in_scope) &
    cost_unconstrained["DestinationID"].isin(java_clusters)
].copy()

print("\nShape cost_lastmile_final (scope đã đúng):", cost_lastmile_final.shape)
coverage_final = cost_lastmile_final.groupby("DestinationID")["OriginID"].nunique()
print(coverage_final.describe())

Tổng facility_model (đã loại Instant Hub): 22
D49/D54 còn trong scope ko: False | False

Shape transfer_final (scope đã đúng, 22 facility): (174, 5)
count    21.000000
mean      8.285714
std       4.496030
min       1.000000
25%       3.000000
50%      12.000000
75%      12.000000
max      12.000000
Name: OriginID, dtype: float64

Shape cost_lastmile_final (scope đã đúng): (10080, 3)
count    1763.000000
mean        5.717527
std         1.534567
min         1.000000
25%         5.000000
50%         6.000000
75%         7.000000
max         9.000000
Name: OriginID, dtype: float64
